# Download the base model

In [ ]:
# no use of DynaSent-R1, DynaSent-R2, or SST-3
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load pre-trained tokenizer and model
model_name = "finiteautomata/bertweet-base-sentiment-analysis"
#model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
model

# Smoke Test

In [ ]:
import torch.nn.functional as F

text = "The movie was fantastic!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
print(inputs)

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits
probs = F.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1).item()
predicted_class

# Dataset Preparation

In [3]:
from datasets import load_dataset

dataset = load_dataset("EnZon3/The-Worlds-Sentiment")

dataset = dataset.rename_column("Sentiment", "score")
dataset = dataset.rename_column("Headline", "sentence")
dataset = dataset.select_columns(['sentence', 'score'])

# Tokenize dataset
def preprocess_sentence(examples):
    return tokenizer(examples['sentence'], padding="max_length", truncation=True)

dataset['train'] = dataset['train'].add_column('labels', [0 if i < 0 else 2 if i > 0 else 1 for i in dataset['train']['score']])

tokenized_dataset = dataset.map(preprocess_sentence, batched=True)
tokenized_dataset = tokenized_dataset['train'].train_test_split(test_size=0.3)

train_dataset = tokenized_dataset['train'].shuffle()
test_dataset = tokenized_dataset['test'].shuffle()

Repo card metadata block was not found. Setting CardData to empty.


# Training

## Prepare the layers

In [ ]:
# Freeze all parameters in the model first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the RobertaIntermediate, RobertaOutput, and RobertaClassificationHead layers
#for layer in model.roberta.encoder.layer:
#    layer.intermediate.requires_grad = True  # Unfreeze RobertaIntermediate layers
#    layer.output.requires_grad = True       # Unfreeze RobertaOutput layers

# Unfreeze the classification head
for param in model.classifier.parameters():
    param.requires_grad = True

# Print out which layers are trainable for verification
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Trainable layer: {name}")
    #else:
    #    print(f"Non-Trainable layer: {name}")

## Train

In [ ]:
from transformers import Trainer, TrainingArguments

# Set up training arguments and Trainer
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    learning_rate=2e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Fine-tune the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)

# Prepare a new model

In [1]:
# no use of DynaSent-R1, DynaSent-R2, or SST-3
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load pre-trained tokenizer and model
model_name = "finiteautomata/bertweet-base-sentiment-analysis"
#model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model2 = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
#for i in model2.named_parameters():
#    print(f"{i[0]} -> {i[1].device}")

C:\Users\sgrebenkin\.conda\envs\cs224u\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:
import torch.nn.functional as F

text = "The movie was fantastic!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

model2.eval()
with torch.no_grad():
    outputs = model2(**inputs)
logits = outputs.logits
probs = F.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1).item()
predicted_class

2

In [4]:
from transformers.models.roberta.modeling_roberta import RobertaLayer

# Add a new RobertaLayer to the encoder's layer list
roberta_encoder2 = model2.roberta.encoder
roberta_encoder2.layer.append(RobertaLayer(model2.config))

# Update the model's configuration
model2.config.num_hidden_layers += 1

# Freeze all existing parameters
for param in model2.parameters():
    param.requires_grad = False

# Unfreeze the newly added layer's parameters
for param in roberta_encoder2.layer[-1].parameters():
    param.requires_grad = True

# Unfreeze the classification head
for param in model2.classifier.parameters():
    param.requires_grad = True

# Print out which layers are trainable for verification
for name, param in model2.named_parameters():
    if param.requires_grad:
        print(f"Trainable layer: {name}")
    #else:
    #    print(f"Non-Trainable layer: {name}")

Trainable layer: roberta.encoder.layer.12.attention.self.query.weight
Trainable layer: roberta.encoder.layer.12.attention.self.query.bias
Trainable layer: roberta.encoder.layer.12.attention.self.key.weight
Trainable layer: roberta.encoder.layer.12.attention.self.key.bias
Trainable layer: roberta.encoder.layer.12.attention.self.value.weight
Trainable layer: roberta.encoder.layer.12.attention.self.value.bias
Trainable layer: roberta.encoder.layer.12.attention.output.dense.weight
Trainable layer: roberta.encoder.layer.12.attention.output.dense.bias
Trainable layer: roberta.encoder.layer.12.attention.output.LayerNorm.weight
Trainable layer: roberta.encoder.layer.12.attention.output.LayerNorm.bias
Trainable layer: roberta.encoder.layer.12.intermediate.dense.weight
Trainable layer: roberta.encoder.layer.12.intermediate.dense.bias
Trainable layer: roberta.encoder.layer.12.output.dense.weight
Trainable layer: roberta.encoder.layer.12.output.dense.bias
Trainable layer: roberta.encoder.layer.12.

In [ ]:
from transformers import Trainer, TrainingArguments

# Set up training arguments and Trainer
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    learning_rate=2e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model2,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Fine-tune the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)

Epoch,Training Loss,Validation Loss
1,No log,1.864450
2,1.922000,1.420435
3,1.922000,1.151830
4,1.097100,0.999025
5,1.097100,0.942031
6,0.914100,0.917656
7,0.914100,0.906533


# Bakeoff

In [ ]:
import os
import wget

if not os.path.exists(os.path.join("data", "sentiment", "cs224u-sentiment-test-unlabeled.csv")):
    os.makedirs(os.path.join('data', 'sentiment'), exist_ok=True)
    wget.download('https://web.stanford.edu/class/cs224u/data/cs224u-sentiment-test-unlabeled.csv', out='data/sentiment/')

In [ ]:
import pandas as pd

bakeoff_df = pd.read_csv(
    os.path.join("data", "sentiment", "cs224u-sentiment-test-unlabeled.csv"))
bakeoff_df

In [ ]:
inputs = tokenizer(bakeoff_df['sentence'].tolist(), return_tensors="pt", truncation=True, padding=True)

# Set to evaluation mode
model2.eval()
with torch.no_grad():
    outputs = model2(**inputs)

In [ ]:
bakeoff_df['prediction'] = torch.argmax(F.softmax(outputs.logits, dim=-1), dim=-1).numpy()
bakeoff_df['prediction'] = bakeoff_df['prediction'].replace(0, 'negative')
bakeoff_df['prediction'] = bakeoff_df['prediction'].replace(1, 'neutral')
bakeoff_df['prediction'] = bakeoff_df['prediction'].replace(2, 'positive')
bakeoff_df.to_csv('predicted/cs224u-sentiment-test-unlabeled.csv')
bakeoff_df